# Projeto AB2 (ICD): preparação dos dados

**Tema:** desigualdades territoriais no rastreamento e no tratamento do câncer de mama no Brasil.

## Objetivo

Este notebook lê a planilha `dados/original/Projeto ICD - Dataset.xlsx` (transcrição de tabelas e figuras do
relatório do INCA *Controle do câncer de mama no Brasil: dados e números 2025*), confere a consistência
dos dados e gera três tabelas tratadas em `dados/tratados/`:

| Arquivo | Conteúdo | Formato |
|---|---|---|
| `base_uf.csv` | um registro por UF (27 linhas) com todos os indicadores e derivados | largo |
| `subgrupos_pns_2019.csv` | mamografia por escolaridade, cor ou raça e rendimento (PNS 2019) | longo |
| `exames_sus_2020_2024.csv` | mamografias de rastreamento (50–69 anos) no SUS, por UF e ano | longo |

Perguntas-guia do projeto:
1. Como variam a incidência e a mortalidade ajustadas entre regiões e UFs (Sul/Sudeste × Norte/Nordeste; Alagoas)?
2. Como varia a cobertura de mamografia (50–69 anos) entre UFs e subgrupos de renda, escolaridade e cor ou raça?
3. Como varia o percentual de casos tratados em até 60 dias entre UFs, e como isso se relaciona com a oferta de mamógrafos e de exames?

Não há modelagem preditiva. A planilha original **não é modificada**; todas as correções acontecem em memória
e ficam registradas aqui e em `docs/relatorio_de_alteracoes.md`.

In [ ]:
# Importações e caminhos relativos ao notebook, para o projeto rodar em qualquer máquina.
from datetime import datetime
from pathlib import Path

import openpyxl
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

RAIZ = Path("..")
ARQ_ORIGINAL = RAIZ / "dados" / "original" / "Projeto ICD - Dataset.xlsx"
DIR_TRATADOS = RAIZ / "dados" / "tratados"
DIR_TRATADOS.mkdir(parents=True, exist_ok=True)
assert ARQ_ORIGINAL.exists(), f"Planilha não encontrada em {ARQ_ORIGINAL.resolve()}"
print("pandas", pd.__version__, "| openpyxl", openpyxl.__version__)

pandas 2.3.3 | openpyxl 3.1.5


In [ ]:
# Constantes usadas em todo o notebook: as 27 UFs e o agrupamento regional pedido pelo projeto.
SIGLAS_UF = sorted([
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
])
assert len(SIGLAS_UF) == 27

GRUPO_COMPARACAO = {
    "Sul": "Sul/Sudeste", "Sudeste": "Sul/Sudeste",
    "Norte": "Norte/Nordeste", "Nordeste": "Norte/Nordeste",
    "Centro-Oeste": "Centro-Oeste",
}

# Nomes das abas e colunas esperados (conferidos contra a planilha logo abaixo).
COMUNS = ["nivel", "regiao", "uf_sigla", "uf_nome", "cod_ibge"]
ABAS_ESPERADAS = {
    "incidencia_mama_inca_2023_2025": COMUNS + ["n_casos", "taxa_bruta", "taxa_ajustada", "periodo", "fonte"],
    "mortalidade_mama_uf_2023": COMUNS + ["ano", "n_obitos", "taxa_bruta", "taxa_ajustada", "fonte"],
    "cobertura_mamografia_uf_pns_201": COMUNS + ["ano", "pct_mamografia_ultimos_2_anos", "fonte"],
    "tratamento_ate_60_dias_uf_2020_": COMUNS + ["periodo", "pct_tratados_ate_60_dias", "classe_mapa", "fonte"],
    "mamografias_rastreamento_50_69_": COMUNS + ["ano", "n_mamografias_rastreamento_50_69", "fonte"],
    "mamografos_uf_cnes_2024": COMUNS + ["ano"]
        + [f"{s}_{t}" for s in ("existentes", "em_uso") for t in ("comando_simples", "estereotaxia", "computadorizado", "total")]
        + ["fonte"],
    "cobertura_mamografia_pns2019_an": COMUNS + ["ano", "dimensao", "categoria", "pct_mamografia_ultimos_2_anos",
                                                 "ic95_inferior", "ic95_superior", "observacao", "fonte"],
    "nunca_fez_mamografia_pns2019_an": COMUNS + ["ano", "dimensao", "categoria", "pct_nunca_fez_mamografia",
                                                 "ic95_inferior", "ic95_superior", "observacao", "fonte"],
}

## 1. Leitura

### Problema encontrado na leitura: números convertidos em datas

Ao abrir a planilha, **303 células numéricas** aparecem como datas (por exemplo, a taxa ajustada de incidência do
Amapá aparece como `20/04/2026`). Quando a tabela foi digitada, o Excel interpretou textos como `20.04` como
"dia 20, mês 04" do ano corrente. Os demais números foram gravados como **texto** (`"16.58"`), por isso
também precisam ser convertidos.

O formato da célula preserva como o número foi digitado, e isso permite recuperar o valor sem ambiguidade:

| Formato da célula | Digitado | Lido como | Valor correto |
|---|---|---|---|
| `dd.mm` | `20.04` | 20/04/2026 | 20,04 |
| `d.m` | `30.6` | 30/06/2026 | 30,6 |
| `d.m` | `7.12` | 07/12/2026 | 7,12 |

Regra: valor = `"{dia}.{mês}"`, com o mês escrito com dois dígitos quando o formato é `mm`. Por isso lemos a
planilha com `openpyxl` (que dá acesso ao formato de cada célula) em vez de `pd.read_excel`.

**Conferência externa:** todos os valores das abas que vêm de tabelas (Tabelas 1, 2, 4, 8, 9 e 13) foram
comparados, após a correção, com o texto extraído do PDF do INCA 2025, e todos coincidem (verificação feita
fora do notebook e descrita no relatório de alterações).

In [ ]:
# Lê cada aba célula a célula para desfazer as datas criadas pelo Excel usando o formato da célula.
def corrigir_data(valor: datetime, formato: str) -> str:
    # Reconstrói o texto digitado: "dd.mm" -> mês com 2 dígitos; "d.m" -> mês como está.
    mes = f"{valor.month:02d}" if "mm" in formato else str(valor.month)
    return f"{valor.day}.{mes}"


def ler_aba(wb, nome):
    ws = wb[nome]
    linhas = list(ws.iter_rows())
    cabecalho = [c.value for c in linhas[0]]
    col_validas = [i for i, h in enumerate(cabecalho) if h is not None]  # ignora colunas vazias à direita
    registros, correcoes = [], []
    for linha in linhas[1:]:
        if all(linha[i].value is None for i in col_validas):
            continue
        reg = {}
        for i in col_validas:
            cel = linha[i]
            valor = cel.value
            if isinstance(valor, datetime):
                assert (valor.hour, valor.minute) == (0, 0) and cel.number_format in ("dd.mm", "d.m"), cel.coordinate
                texto = corrigir_data(valor, cel.number_format)
                correcoes.append({"aba": nome, "celula": cel.coordinate, "coluna": cabecalho[i],
                                  "lido_como": valor.date(), "formato": cel.number_format,
                                  "valor_corrigido": float(texto)})
                valor = texto
            reg[cabecalho[i]] = valor
        registros.append(reg)
    return pd.DataFrame(registros), correcoes


wb = openpyxl.load_workbook(ARQ_ORIGINAL)  # leitura apenas; o arquivo nunca é salvo
abas, log_datas = {}, []
for nome in wb.sheetnames:
    abas[nome], corr = ler_aba(wb, nome)
    log_datas += corr
log_datas = pd.DataFrame(log_datas)

print("Abas encontradas:", wb.sheetnames)
for nome, df in abas.items():
    print(f"  {nome:35s} {df.shape}")

Abas encontradas: ['incidencia_mama_inca_2023_2025', 'mamografias_rastreamento_50_69_', 'tratamento_ate_60_dias_uf_2020_', 'mortalidade_mama_uf_2023', 'cobertura_mamografia_pns2019_an', 'cobertura_mamografia_uf_pns_201', 'nunca_fez_mamografia_pns2019_an', 'mamografos_uf_cnes_2024']
  incidencia_mama_inca_2023_2025      (33, 10)
  mamografias_rastreamento_50_69_     (165, 8)
  tratamento_ate_60_dias_uf_2020_     (27, 9)
  mortalidade_mama_uf_2023            (33, 10)
  cobertura_mamografia_pns2019_an     (462, 13)
  cobertura_mamografia_uf_pns_201     (56, 8)
  nunca_fez_mamografia_pns2019_an     (90, 13)
  mamografos_uf_cnes_2024             (33, 15)


In [ ]:
# Confirma que nomes de abas e colunas são os esperados antes de seguir.
assert set(abas) == set(ABAS_ESPERADAS), set(abas) ^ set(ABAS_ESPERADAS)
for nome, esperadas in ABAS_ESPERADAS.items():
    assert list(abas[nome].columns) == esperadas, (nome, list(abas[nome].columns))
print("Nomes de abas e colunas conferem com o esperado.")

Nomes de abas e colunas conferem com o esperado.


In [ ]:
# Resume as células corrigidas (data -> número) para registro no relatório de alterações.
print("Total de células corrigidas:", len(log_datas))
display(log_datas.groupby(["aba", "coluna"]).size().rename("celulas").to_frame())
display(log_datas.head(8))

Total de células corrigidas: 303


celulas
aba                             coluna                                
cobertura_mamografia_pns2019_an ic95_inferior                      101
                                ic95_superior                        1
                                pct_mamografia_ultimos_2_anos       22
incidencia_mama_inca_2023_2025  taxa_ajustada                        1
                                taxa_bruta                           1
mortalidade_mama_uf_2023        taxa_ajustada                        6
                                taxa_bruta                           2
nunca_fez_mamografia_pns2019_an ic95_inferior                       66
                                ic95_superior                       44
                                pct_nunca_fez_mamografia            58
tratamento_ate_60_dias_uf_2020_ pct_tratados_ate_60_dias             1

,aba,celula,coluna,lido_como,formato,valor_corrigido
0,incidencia_mama_inca_2023_2025,H4,taxa_ajustada,2026-04-20,dd.mm,20.04
1,incidencia_mama_inca_2023_2025,G8,taxa_bruta,2026-09-22,dd.mm,22.09
2,tratamento_ate_60_dias_uf_2020_,G3,pct_tratados_ate_60_dias,2026-06-30,d.m,30.60
3,mortalidade_mama_uf_2023,H3,taxa_bruta,2026-09-07,dd.mm,7.09
4,mortalidade_mama_uf_2023,I4,taxa_ajustada,2026-12-07,d.m,7.12
5,mortalidade_mama_uf_2023,I6,taxa_ajustada,2026-09-08,dd.mm,8.09
6,mortalidade_mama_uf_2023,I17,taxa_ajustada,2026-12-09,d.m,9.12
7,mortalidade_mama_uf_2023,I21,taxa_ajustada,2026-08-12,dd.mm,12.08


In [ ]:
# Converte colunas numéricas (gravadas como texto) para número; errors="raise" impede conversões silenciosas.
COLS_INTEIRAS = ["cod_ibge", "ano", "n_casos", "n_obitos", "n_mamografias_rastreamento_50_69"] + \
    [c for c in ABAS_ESPERADAS["mamografos_uf_cnes_2024"] if c.startswith(("existentes_", "em_uso_"))]
COLS_DECIMAIS = ["taxa_bruta", "taxa_ajustada", "pct_mamografia_ultimos_2_anos", "pct_tratados_ate_60_dias",
                 "pct_nunca_fez_mamografia", "ic95_inferior", "ic95_superior"]

for nome, df in abas.items():
    for c in df.columns:
        if c in COLS_INTEIRAS:
            df[c] = pd.to_numeric(df[c], errors="raise").astype("Int64")
        elif c in COLS_DECIMAIS:
            df[c] = pd.to_numeric(df[c], errors="raise").astype(float)
    # Garante que só sobraram textos onde se espera texto.
    obj = [c for c in df.columns if df[c].dtype == object]
    assert set(obj) <= {"nivel", "regiao", "uf_sigla", "uf_nome", "periodo", "classe_mapa", "fonte",
                        "dimensao", "categoria", "observacao"}, (nome, obj)

# Atalhos com nomes curtos para as abas.
inc = abas["incidencia_mama_inca_2023_2025"]
mort = abas["mortalidade_mama_uf_2023"]
cob_uf = abas["cobertura_mamografia_uf_pns_201"]
trat = abas["tratamento_ate_60_dias_uf_2020_"]
exames = abas["mamografias_rastreamento_50_69_"]
mamog = abas["mamografos_uf_cnes_2024"]
sub_fez = abas["cobertura_mamografia_pns2019_an"]
sub_nunca = abas["nunca_fez_mamografia_pns2019_an"]
inc.dtypes

nivel             object
regiao            object
uf_sigla          object
uf_nome           object
cod_ibge           Int64
n_casos            Int64
taxa_bruta       float64
taxa_ajustada    float64
periodo           object
fonte             object
dtype: object

## 2. Validação

Conferências feitas antes de qualquer junção. Todas são `assert`: se a planilha mudar e uma delas falhar, o
notebook para com uma mensagem indicando o problema.

In [ ]:
# (a) Toda aba com nível UF tem exatamente as 27 siglas, sem duplicatas (por ano/subgrupo quando houver).
chaves_extra = {
    "cobertura_mamografia_uf_pns_201": ["ano"],
    "mamografias_rastreamento_50_69_": ["ano"],
    "cobertura_mamografia_pns2019_an": ["dimensao", "categoria"],
}
for nome, df in abas.items():
    ufs = df[df["nivel"] == "uf"]
    if ufs.empty:
        print(f"  {nome}: sem linhas de UF (só região e Brasil), como esperado")
        assert nome == "nunca_fez_mamografia_pns2019_an"
        continue
    extra = chaves_extra.get(nome, [])
    assert not ufs.duplicated(["uf_sigla"] + extra).any(), f"duplicatas em {nome}"
    grupos = ufs.groupby(extra) if extra else [(None, ufs)]
    for _, g in grupos:
        assert sorted(g["uf_sigla"]) == SIGLAS_UF, f"{nome}: UFs faltando ou sobrando"
    print(f"  {nome}: 27 UFs em cada um dos {len(grupos) if extra else 1} recortes")
assert set(trat["nivel"]) == {"uf"}  # tratamento só tem UF
assert set(sub_nunca["nivel"]) == {"regiao", "pais"}

  incidencia_mama_inca_2023_2025: 27 UFs em cada um dos 1 recortes
  mamografias_rastreamento_50_69_: 27 UFs em cada um dos 5 recortes
  tratamento_ate_60_dias_uf_2020_: 27 UFs em cada um dos 1 recortes
  mortalidade_mama_uf_2023: 27 UFs em cada um dos 1 recortes
  cobertura_mamografia_pns2019_an: 27 UFs em cada um dos 14 recortes
  cobertura_mamografia_uf_pns_201: 27 UFs em cada um dos 2 recortes
  nunca_fez_mamografia_pns2019_an: sem linhas de UF (só região e Brasil), como esperado
  mamografos_uf_cnes_2024: 27 UFs em cada um dos 1 recortes


In [ ]:
# (b) Identificação das UFs é a mesma em todas as abas (sigla -> nome, código IBGE e região).
ident = pd.concat([df.loc[df["nivel"] == "uf", ["uf_sigla", "uf_nome", "cod_ibge", "regiao"]] for df in abas.values()])
ident = ident.drop_duplicates()
assert len(ident) == 27 and ident["uf_sigla"].is_unique, ident[ident["uf_sigla"].duplicated(keep=False)]
assert set(ident["regiao"]) == set(GRUPO_COMPARACAO)
# Linhas de região/Brasil não têm sigla, e todas as abas usam os mesmos rótulos de nível.
for nome, df in abas.items():
    assert set(df["nivel"]) <= {"uf", "regiao", "pais"}
    assert df.loc[df["nivel"] != "uf", "uf_sigla"].isna().all(), nome
print("Identificação das UFs consistente entre as 8 abas.")

Identificação das UFs consistente entre as 8 abas.


In [ ]:
# (c) Nas abas com totais, a soma das UFs reproduz cada região e o Brasil (e a soma das regiões, o Brasil).
def conferir_totais(df, cols, por=()):
    por = list(por)
    uf = df[df["nivel"] == "uf"]
    reg = df[df["nivel"] == "regiao"].set_index(["regiao"] + por)[cols].sort_index()
    pais = df[df["nivel"] == "pais"]
    pais = (pais.set_index(por)[cols] if por else pais[cols].reset_index(drop=True)).sort_index()
    soma_uf_reg = uf.groupby(["regiao"] + por)[cols].sum().sort_index()
    pd.testing.assert_frame_equal(soma_uf_reg, reg, check_dtype=False)
    for nivel_soma in (uf, df[df["nivel"] == "regiao"]):
        soma = nivel_soma.groupby(por)[cols].sum() if por else nivel_soma[cols].sum().to_frame().T
        assert (soma.sort_index().to_numpy() == pais.to_numpy()).all()

conferir_totais(inc, ["n_casos"])
conferir_totais(mort, ["n_obitos"])
conferir_totais(exames, ["n_mamografias_rastreamento_50_69"], por=["ano"])
COLS_MAMOG = [c for c in mamog.columns if c.startswith(("existentes_", "em_uso_"))]
conferir_totais(mamog, COLS_MAMOG)
print("Totais conferem: incidência (casos), mortalidade (óbitos), exames (5 anos) e mamógrafos (8 colunas).")

Totais conferem: incidência (casos), mortalidade (óbitos), exames (5 anos) e mamógrafos (8 colunas).


In [ ]:
# (d) Mamógrafos: tipos somam o total e equipamentos em uso nunca superam os existentes.
TIPOS = ["comando_simples", "estereotaxia", "computadorizado"]
for s in ("existentes", "em_uso"):
    assert (mamog[[f"{s}_{t}" for t in TIPOS]].sum(axis=1) == mamog[f"{s}_total"]).all(), s
for t in TIPOS + ["total"]:
    assert (mamog[f"em_uso_{t}"] <= mamog[f"existentes_{t}"]).all(), t
print("Mamógrafos: tipos = total e em uso ≤ existentes em todas as 33 linhas.")

Mamógrafos: tipos = total e em uso ≤ existentes em todas as 33 linhas.


In [ ]:
# (e) Subgrupos PNS: quando há IC, ic95_inferior ≤ percentual ≤ ic95_superior; IC ausente só onde a fonte indica.
for nome, col in [("cobertura_mamografia_pns2019_an", "pct_mamografia_ultimos_2_anos"),
                  ("nunca_fez_mamografia_pns2019_an", "pct_nunca_fez_mamografia")]:
    df = abas[nome]
    com_ic = df["ic95_inferior"].notna() & df["ic95_superior"].notna()
    ok = (df.loc[com_ic, "ic95_inferior"] <= df.loc[com_ic, col]) & (df.loc[com_ic, col] <= df.loc[com_ic, "ic95_superior"])
    assert ok.all(), df.loc[com_ic][~ok]
    sem_ic = df[~com_ic]
    assert (sem_ic["observacao"] == "IC sem limites na fonte").all(), sem_ic
    print(f"  {nome}: {com_ic.sum()} linhas com IC coerente; {len(sem_ic)} sem IC")
display(sub_fez[sub_fez["ic95_inferior"].isna()][["uf_sigla", "dimensao", "categoria", "pct_mamografia_ultimos_2_anos", "observacao"]])

  cobertura_mamografia_pns2019_an: 461 linhas com IC coerente; 1 sem IC
  nunca_fez_mamografia_pns2019_an: 90 linhas com IC coerente; 0 sem IC


,uf_sigla,dimensao,categoria,pct_mamografia_ultimos_2_anos,observacao
180,MA,rendimento,De 3 a 5 SM,100.0,IC sem limites na fonte


In [ ]:
# (f) Faixas plausíveis: percentuais em [0, 100]; taxas e contagens > 0; sem valores ausentes nos indicadores.
percentuais = [(cob_uf, "pct_mamografia_ultimos_2_anos"), (trat, "pct_tratados_ate_60_dias"),
               (sub_fez, "pct_mamografia_ultimos_2_anos"), (sub_fez, "ic95_inferior"), (sub_fez, "ic95_superior"),
               (sub_nunca, "pct_nunca_fez_mamografia"), (sub_nunca, "ic95_inferior"), (sub_nunca, "ic95_superior")]
for df, c in percentuais:
    v = df[c].dropna()
    assert v.between(0, 100).all(), c
positivos = [(inc, "n_casos"), (inc, "taxa_bruta"), (inc, "taxa_ajustada"),
             (mort, "n_obitos"), (mort, "taxa_bruta"), (mort, "taxa_ajustada"),
             (exames, "n_mamografias_rastreamento_50_69")] + [(mamog, c) for c in COLS_MAMOG]
for df, c in positivos:
    assert df[c].notna().all() and (df[c] > 0).all(), c
for df, c in percentuais[:2] + [(sub_fez, "pct_mamografia_ultimos_2_anos"), (sub_nunca, "pct_nunca_fez_mamografia")]:
    assert df[c].notna().all(), c
print("Percentuais em [0, 100]; taxas e contagens positivas e completas.")

Percentuais em [0, 100]; taxas e contagens positivas e completas.


### Problemas conhecidos da fonte (documentados, não corrigidos)

* **Incidência:** as taxas ajustadas de três regiões são idênticas às de uma UF da própria região
  (Norte = Roraima 27,73; Nordeste = Sergipe 42,11; Sul = Paraná 41,06). O mesmo aparece no PDF, então é
  provável erro do relatório, não da transcrição. Essas taxas regionais **não são usadas**: a `base_uf` só
  traz UFs, e qualquer comparação regional de incidência ajustada deve ser evitada ou feita com ressalva.
* **PNS (subgrupos):** dígitos soltos impressos junto de alguns valores (registrados na `observacao`);
  valores em cinza para as UFs do Centro-Oeste em "Médio completo e superior incompleto"; Maranhão
  "100% sem limites" (excluído, ver seção 4).
* **Tratamento:** o INCA excluiu da Figura 26 os casos sem informação de tratamento.
* **Períodos diferentes** entre indicadores: incidência 2023–2025 (estimativa anual), mortalidade 2023,
  PNS 2013/2019, exames 2020–2024, tratamento 2020–2024, mamógrafos dez/2024.

In [ ]:
# Mostra, a partir dos dados, as três coincidências de taxa ajustada região = UF citadas acima.
reg = inc[inc["nivel"] == "regiao"][["regiao", "taxa_ajustada"]]
uf = inc[inc["nivel"] == "uf"][["regiao", "uf_sigla", "taxa_ajustada"]]
coinc = reg.merge(uf, on=["regiao", "taxa_ajustada"])
display(coinc)
assert set(zip(coinc["regiao"], coinc["uf_sigla"])) == {("Norte", "RR"), ("Nordeste", "SE"), ("Sul", "PR")}

,regiao,taxa_ajustada,uf_sigla
0,Norte,27.73,RR
1,Nordeste,42.11,SE
2,Sul,41.06,PR


## 3. Montagem da `base_uf` (uma linha por UF)

Partimos das 27 UFs da aba de incidência e juntamos cada indicador pela sigla. Cada junção usa
`validate="one_to_one"` e `how="left"`, e no final conferimos que não surgiram nulos.

In [ ]:
# Identificação + grupo de comparação (Sul/Sudeste, Norte/Nordeste, Centro-Oeste).
base = (inc[inc["nivel"] == "uf"][["uf_sigla", "uf_nome", "cod_ibge", "regiao"]]
        .assign(grupo_comparacao=lambda d: d["regiao"].map(GRUPO_COMPARACAO))
        .reset_index(drop=True))
assert base["grupo_comparacao"].notna().all()
base.head()

,uf_sigla,uf_nome,cod_ibge,regiao,grupo_comparacao
0,AC,Acre,12,Norte,Norte/Nordeste
1,AP,Amapá,16,Norte,Norte/Nordeste
2,AM,Amazonas,13,Norte,Norte/Nordeste
3,PA,Pará,15,Norte,Norte/Nordeste
4,RO,Rondônia,11,Norte,Norte/Nordeste


In [ ]:
# Seleciona e renomeia cada indicador para o nível UF; nomes com período quando ele não é óbvio.
so_uf = lambda df: df[df["nivel"] == "uf"]

t_inc = so_uf(inc)[["uf_sigla", "n_casos", "taxa_bruta", "taxa_ajustada"]].rename(columns={
    "n_casos": "incid_casos", "taxa_bruta": "incid_taxa_bruta", "taxa_ajustada": "incid_taxa_ajustada"})

assert (so_uf(mort)["ano"] == 2023).all()
t_mort = so_uf(mort)[["uf_sigla", "n_obitos", "taxa_bruta", "taxa_ajustada"]].rename(columns={
    "n_obitos": "mort_obitos_2023", "taxa_bruta": "mort_taxa_bruta_2023", "taxa_ajustada": "mort_taxa_ajustada_2023"})

# Cobertura PNS: formato longo (uma linha por UF e ano) -> largo (uma coluna por ano).
t_cob = (so_uf(cob_uf).pivot(index="uf_sigla", columns="ano", values="pct_mamografia_ultimos_2_anos")
         .rename(columns={2013: "cobertura_pns_2013", 2019: "cobertura_pns_2019"}).reset_index())
t_cob.columns.name = None

t_trat = trat[["uf_sigla", "pct_tratados_ate_60_dias"]]

t_exames = (so_uf(exames).query("ano == 2024")[["uf_sigla", "n_mamografias_rastreamento_50_69"]]
            .rename(columns={"n_mamografias_rastreamento_50_69": "exames_sus_2024"}))

assert (so_uf(mamog)["ano"] == 2024).all()
t_mamog = so_uf(mamog)[["uf_sigla", "existentes_total", "em_uso_total"]].rename(columns={
    "existentes_total": "mamografos_existentes_2024", "em_uso_total": "mamografos_em_uso_2024"})

for t in (t_inc, t_mort, t_cob, t_trat, t_exames, t_mamog):
    base = base.merge(t, on="uf_sigla", how="left", validate="one_to_one")
assert len(base) == 27 and base.notna().all().all()
base.head()

,uf_sigla,uf_nome,cod_ibge,regiao,grupo_comparacao,incid_casos,incid_taxa_bruta,incid_taxa_ajustada,mort_obitos_2023,mort_taxa_bruta_2023,mort_taxa_ajustada_2023,cobertura_pns_2013,cobertura_pns_2019,pct_tratados_ate_60_dias,exames_sus_2024,mamografos_existentes_2024,mamografos_em_uso_2024
0,AC,Acre,12,Norte,Norte/Nordeste,100,22.21,26.20,31,7.09,7.17,36.9,40.2,39.0,7649,26,25
1,AP,Amapá,16,Norte,Norte/Nordeste,80,16.58,20.04,26,6.51,7.12,38.0,34.9,30.6,3613,26,23
2,AM,Amazonas,13,Norte,Norte/Nordeste,500,22.77,28.34,253,12.00,12.25,52.4,51.5,34.1,20943,128,114
3,PA,Pará,15,Norte,Norte/Nordeste,1020,22.83,23.88,382,8.91,8.09,35.8,40.6,42.0,37334,201,189
4,RO,Rondônia,11,Norte,Norte/Nordeste,320,35.33,36.99,118,13.65,11.53,34.8,44.0,42.4,9274,67,61


## 4. Variáveis derivadas

| Variável | Fórmula | Observação |
|---|---|---|
| `pop_feminina_estimada` | `incid_casos / incid_taxa_bruta × 100.000` | Aproximada: população feminina de **todas as idades** implícita na estimativa do INCA 2023–2025. Arredondada para inteiro. Como `incid_casos` é arredondado pelo INCA (múltiplos de 10), o erro é maior nas UFs pequenas. |
| `mamografos_por_100mil_mulheres` | `mamografos_em_uso_2024 / pop_feminina_estimada × 100.000` | Usa a população não arredondada; resultado com 2 casas. O denominador é de todas as idades, não só 50–69 anos. |
| `var_cobertura_2013_2019` | `cobertura_pns_2019 − cobertura_pns_2013` | Pontos percentuais (p.p.), 1 casa. |
| `razao_mortalidade_incidencia` | `mort_taxa_ajustada_2023 / incid_taxa_ajustada` | Adimensional, 3 casas. Indicador grosseiro: os períodos diferem (2023 × 2023–2025). |

In [ ]:
# Calcula os derivados a partir das colunas já validadas; o arredondamento é só para gravação legível.
pop = base["incid_casos"] / base["incid_taxa_bruta"] * 100_000
base["pop_feminina_estimada"] = pop.round(0).astype("Int64")
base["mamografos_por_100mil_mulheres"] = (base["mamografos_em_uso_2024"] / pop * 100_000).astype(float).round(2)
base["var_cobertura_2013_2019"] = (base["cobertura_pns_2019"] - base["cobertura_pns_2013"]).round(1)
base["razao_mortalidade_incidencia"] = (base["mort_taxa_ajustada_2023"] / base["incid_taxa_ajustada"]).round(3)

# Conferência de ordem de grandeza: a soma das populações estimadas fica perto da população feminina do Brasil
# implícita na linha "Brasil" da mesma tabela (diferença vem do arredondamento dos casos).
pop_brasil = (inc.loc[inc["nivel"] == "pais", "n_casos"] / inc.loc[inc["nivel"] == "pais", "taxa_bruta"] * 100_000).iloc[0]
print(f"Soma das UFs: {pop.sum():,.0f} | Brasil implícito: {pop_brasil:,.0f} | diferença: {pop.sum() / pop_brasil - 1:+.2%}")
assert abs(pop.sum() / pop_brasil - 1) < 0.01
base[["uf_sigla", "pop_feminina_estimada", "mamografos_por_100mil_mulheres",
      "var_cobertura_2013_2019", "razao_mortalidade_incidencia"]].head()

Soma das UFs: 110,635,860 | Brasil implícito: 110,625,188 | diferença: +0.01%


,uf_sigla,pop_feminina_estimada,mamografos_por_100mil_mulheres,var_cobertura_2013_2019,razao_mortalidade_incidencia
0,AC,450248,5.55,3.3,0.274
1,AP,482509,4.77,-3.1,0.355
2,AM,2195872,5.19,-0.9,0.432
3,PA,4467806,4.23,4.8,0.339
4,RO,905746,6.73,9.2,0.312


In [ ]:
# Ordena as colunas como no dicionário de dados e as linhas por região e sigla (estável entre execuções).
ORDEM_BASE = [
    "uf_sigla", "uf_nome", "cod_ibge", "regiao", "grupo_comparacao",
    "incid_casos", "incid_taxa_bruta", "incid_taxa_ajustada",
    "mort_obitos_2023", "mort_taxa_bruta_2023", "mort_taxa_ajustada_2023",
    "cobertura_pns_2013", "cobertura_pns_2019",
    "pct_tratados_ate_60_dias",
    "exames_sus_2024",
    "mamografos_existentes_2024", "mamografos_em_uso_2024",
    "pop_feminina_estimada", "mamografos_por_100mil_mulheres", "var_cobertura_2013_2019",
    "razao_mortalidade_incidencia",
]
ORDEM_REGIOES = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
assert set(ORDEM_BASE) == set(base.columns)
base_uf = (base[ORDEM_BASE]
           .assign(_r=lambda d: d["regiao"].map(ORDEM_REGIOES.index))
           .sort_values(["_r", "uf_sigla"]).drop(columns="_r").reset_index(drop=True))

# Validações finais da tabela principal.
assert base_uf.shape == (27, len(ORDEM_BASE))
assert base_uf.notna().all().all(), base_uf.isna().sum()
assert sorted(base_uf["uf_sigla"]) == SIGLAS_UF
base_uf.head()

,uf_sigla,uf_nome,cod_ibge,regiao,grupo_comparacao,incid_casos,incid_taxa_bruta,incid_taxa_ajustada,mort_obitos_2023,mort_taxa_bruta_2023,mort_taxa_ajustada_2023,cobertura_pns_2013,cobertura_pns_2019,pct_tratados_ate_60_dias,exames_sus_2024,mamografos_existentes_2024,mamografos_em_uso_2024,pop_feminina_estimada,mamografos_por_100mil_mulheres,var_cobertura_2013_2019,razao_mortalidade_incidencia
0,AC,Acre,12,Norte,Norte/Nordeste,100,22.21,26.20,31,7.09,7.17,36.9,40.2,39.0,7649,26,25,450248,5.55,3.3,0.274
1,AM,Amazonas,13,Norte,Norte/Nordeste,500,22.77,28.34,253,12.00,12.25,52.4,51.5,34.1,20943,128,114,2195872,5.19,-0.9,0.432
2,AP,Amapá,16,Norte,Norte/Nordeste,80,16.58,20.04,26,6.51,7.12,38.0,34.9,30.6,3613,26,23,482509,4.77,-3.1,0.355
3,PA,Pará,15,Norte,Norte/Nordeste,1020,22.83,23.88,382,8.91,8.09,35.8,40.6,42.0,37334,201,189,4467806,4.23,4.8,0.339
4,RO,Rondônia,11,Norte,Norte/Nordeste,320,35.33,36.99,118,13.65,11.53,34.8,44.0,42.4,9274,67,61,905746,6.73,9.2,0.312


## 5. `subgrupos_pns_2019` (formato longo)

Empilha as duas abas de subgrupos (Tabelas 8 e 9 do INCA) com uma coluna `indicador`:

* `fez_ultimos_2_anos`: mamografia há menos de dois anos (UF, região e Brasil; 14 subgrupos);
* `nunca_fez`: nunca fez mamografia (só região e Brasil; 14 subgrupos + "Total").

A linha do **Maranhão, rendimento "De 3 a 5 SM"** (100%, "IC sem limites na fonte") é excluída: sem intervalo
de confiança, o valor de 100% indica amostra muito pequena e não é comparável com os demais.

In [ ]:
# Padroniza as duas abas para o mesmo esquema e empilha (formato longo).
COLS_SUB = ["nivel", "regiao", "uf_sigla", "indicador", "dimensao", "categoria",
            "percentual", "ic95_inferior", "ic95_superior", "observacao"]
partes = [
    sub_fez.rename(columns={"pct_mamografia_ultimos_2_anos": "percentual"}).assign(indicador="fez_ultimos_2_anos"),
    sub_nunca.rename(columns={"pct_nunca_fez_mamografia": "percentual"}).assign(indicador="nunca_fez"),
]
assert all((p["ano"] == 2019).all() for p in partes)
subgrupos = pd.concat([p[COLS_SUB] for p in partes], ignore_index=True)
print("Linhas antes da exclusão:", len(subgrupos))

Linhas antes da exclusão: 552


In [ ]:
# Exclui a linha "IC sem limites" do Maranhão e confere que foi exatamente uma linha.
mask_excluir = (subgrupos["uf_sigla"] == "MA") & (subgrupos["observacao"] == "IC sem limites na fonte")
excluidas = subgrupos[mask_excluir]
display(excluidas)
assert len(excluidas) == 1
subgrupos_pns = subgrupos[~mask_excluir].reset_index(drop=True)

# Depois da exclusão, todas as linhas têm IC e o IC contém o percentual.
assert subgrupos_pns[["percentual", "ic95_inferior", "ic95_superior"]].notna().all().all()
assert ((subgrupos_pns["ic95_inferior"] <= subgrupos_pns["percentual"]) &
        (subgrupos_pns["percentual"] <= subgrupos_pns["ic95_superior"])).all()
assert not subgrupos_pns.duplicated(["nivel", "regiao", "uf_sigla", "indicador", "dimensao", "categoria"]).any()
print(subgrupos_pns.groupby(["indicador", "nivel"]).size())
print("\nObservações mantidas como na fonte:")
print(subgrupos_pns["observacao"].value_counts())

,nivel,regiao,uf_sigla,indicador,dimensao,categoria,percentual,ic95_inferior,ic95_superior,observacao
180,uf,Nordeste,MA,fez_ultimos_2_anos,rendimento,De 3 a 5 SM,100.0,NaN,NaN,IC sem limites na fonte


indicador           nivel 
fez_ultimos_2_anos  pais       14
                    regiao     70
                    uf        377
nunca_fez           pais       15
                    regiao     75
dtype: int64

Observações mantidas como na fonte:
observacao
Valor em cinza na fonte                                                          4
Na fonte aparece um dígito extra junto ao valor ("13,7 / 7"); registrado 13,7    1
Na fonte aparece um dígito extra junto ao valor ("23,7 / 7"); registrado 23,7    1
Na fonte aparece um dígito extra junto ao valor ("15,1 / 1"); registrado 15,1    1
Na fonte aparece um dígito extra junto ao valor ("15,4 / 4"); registrado 15,4    1
Na fonte aparece um dígito extra junto ao valor ("4,9 / 9"); registrado 4,9      1
Na fonte aparece um dígito extra junto ao valor ("3,8 / 8"); registrado 3,8      1
Na fonte aparece um dígito extra junto ao valor ("4,8 / 8"); registrado 4,8      1
Na fonte aparece um dígito extra junto ao valor ("7,2 / 2"); registrado 7,2   

## 6. `exames_sus_2020_2024` (formato longo)

Apenas as linhas de UF da Tabela 4 (mamografias de rastreamento em mulheres de 50 a 69 anos no SUS), com a
região e o grupo de comparação para facilitar os gráficos.

In [ ]:
# Filtra UFs, acrescenta o grupo de comparação e ordena por região, UF e ano.
exames_sus = (so_uf(exames)
              .assign(grupo_comparacao=lambda d: d["regiao"].map(GRUPO_COMPARACAO),
                      _r=lambda d: d["regiao"].map(ORDEM_REGIOES.index))
              .sort_values(["_r", "uf_sigla", "ano"])
              [["uf_sigla", "uf_nome", "regiao", "grupo_comparacao", "ano", "n_mamografias_rastreamento_50_69"]]
              .reset_index(drop=True))
assert len(exames_sus) == 27 * 5 and sorted(exames_sus["ano"].unique()) == [2020, 2021, 2022, 2023, 2024]
assert exames_sus.notna().all().all() and not exames_sus.duplicated(["uf_sigla", "ano"]).any()
# A coluna exames_sus_2024 da base_uf deve bater com esta tabela.
chk = exames_sus.query("ano == 2024").set_index("uf_sigla")["n_mamografias_rastreamento_50_69"]
assert (base_uf.set_index("uf_sigla")["exames_sus_2024"] == chk.reindex(base_uf["uf_sigla"])).all()
exames_sus.head()

,uf_sigla,uf_nome,regiao,grupo_comparacao,ano,n_mamografias_rastreamento_50_69
0,AC,Acre,Norte,Norte/Nordeste,2020,1690
1,AC,Acre,Norte,Norte/Nordeste,2021,1641
2,AC,Acre,Norte,Norte/Nordeste,2022,5000
3,AC,Acre,Norte,Norte/Nordeste,2023,6225
4,AC,Acre,Norte,Norte/Nordeste,2024,7649


## 7. Gravação

CSV em UTF-8, separador vírgula e ponto decimal, sem índice. Valores ausentes (só na coluna `observacao`)
ficam como campo vazio.

In [ ]:
# Grava as três saídas e relê cada arquivo para confirmar que o conteúdo foi preservado.
SAIDAS = {
    "base_uf.csv": base_uf,
    "subgrupos_pns_2019.csv": subgrupos_pns,
    "exames_sus_2020_2024.csv": exames_sus,
}
for arquivo, df in SAIDAS.items():
    caminho = DIR_TRATADOS / arquivo
    df.to_csv(caminho, index=False, encoding="utf-8")
    relido = pd.read_csv(caminho, encoding="utf-8")
    assert relido.shape == df.shape and list(relido.columns) == list(df.columns), arquivo
    print(f"  gravado {caminho.as_posix():45s} {df.shape}")

  gravado ../dados/tratados/base_uf.csv                 (27, 21)
  gravado ../dados/tratados/subgrupos_pns_2019.csv      (551, 10)
  gravado ../dados/tratados/exames_sus_2020_2024.csv    (135, 6)


## 8. Resumo final

In [ ]:
# Shape e primeiras linhas de cada saída, para conferência visual.
for arquivo, df in SAIDAS.items():
    print(f"\n### {arquivo}  shape={df.shape}")
    display(df.head())


### base_uf.csv  shape=(27, 21)


,uf_sigla,uf_nome,cod_ibge,regiao,grupo_comparacao,incid_casos,incid_taxa_bruta,incid_taxa_ajustada,mort_obitos_2023,mort_taxa_bruta_2023,mort_taxa_ajustada_2023,cobertura_pns_2013,cobertura_pns_2019,pct_tratados_ate_60_dias,exames_sus_2024,mamografos_existentes_2024,mamografos_em_uso_2024,pop_feminina_estimada,mamografos_por_100mil_mulheres,var_cobertura_2013_2019,razao_mortalidade_incidencia
0,AC,Acre,12,Norte,Norte/Nordeste,100,22.21,26.20,31,7.09,7.17,36.9,40.2,39.0,7649,26,25,450248,5.55,3.3,0.274
1,AM,Amazonas,13,Norte,Norte/Nordeste,500,22.77,28.34,253,12.00,12.25,52.4,51.5,34.1,20943,128,114,2195872,5.19,-0.9,0.432
2,AP,Amapá,16,Norte,Norte/Nordeste,80,16.58,20.04,26,6.51,7.12,38.0,34.9,30.6,3613,26,23,482509,4.77,-3.1,0.355
3,PA,Pará,15,Norte,Norte/Nordeste,1020,22.83,23.88,382,8.91,8.09,35.8,40.6,42.0,37334,201,189,4467806,4.23,4.8,0.339
4,RO,Rondônia,11,Norte,Norte/Nordeste,320,35.33,36.99,118,13.65,11.53,34.8,44.0,42.4,9274,67,61,905746,6.73,9.2,0.312



### subgrupos_pns_2019.csv  shape=(551, 10)


,nivel,regiao,uf_sigla,indicador,dimensao,categoria,percentual,ic95_inferior,ic95_superior,observacao
0,regiao,Norte,None,fez_ultimos_2_anos,escolaridade,Sem instrução e fundamental incompleto,32.5,28.8,36.4,None
1,regiao,Norte,None,fez_ultimos_2_anos,escolaridade,Fundamental completo e médio incompleto,49.6,41.3,57.8,None
2,regiao,Norte,None,fez_ultimos_2_anos,escolaridade,Médio completo e superior incompleto,52.3,46.3,58.2,None
3,regiao,Norte,None,fez_ultimos_2_anos,escolaridade,Superior completo,65.1,56.4,72.9,None
4,regiao,Norte,None,fez_ultimos_2_anos,cor_raca,Branca,50.4,43.7,57.2,None



### exames_sus_2020_2024.csv  shape=(135, 6)


,uf_sigla,uf_nome,regiao,grupo_comparacao,ano,n_mamografias_rastreamento_50_69
0,AC,Acre,Norte,Norte/Nordeste,2020,1690
1,AC,Acre,Norte,Norte/Nordeste,2021,1641
2,AC,Acre,Norte,Norte/Nordeste,2022,5000
3,AC,Acre,Norte,Norte/Nordeste,2023,6225
4,AC,Acre,Norte,Norte/Nordeste,2024,7649


In [ ]:
# Visão rápida de Alagoas frente às médias simples (não ponderadas) das UFs de cada grupo; só leitura, sem conclusões.
cols = ["incid_taxa_ajustada", "mort_taxa_ajustada_2023", "cobertura_pns_2019", "pct_tratados_ate_60_dias",
        "mamografos_por_100mil_mulheres"]
resumo = base_uf.groupby("grupo_comparacao")[cols].mean().round(2)
resumo.loc["Alagoas"] = base_uf.loc[base_uf["uf_sigla"] == "AL", cols].iloc[0]
resumo

,incid_taxa_ajustada,mort_taxa_ajustada_2023,cobertura_pns_2019,pct_tratados_ate_60_dias,mamografos_por_100mil_mulheres
grupo_comparacao,,,,,
Centro-Oeste,47.50,12.45,57.10,38.28,8.27
Norte/Nordeste,36.34,10.70,45.78,41.31,6.42
Sul/Sudeste,51.66,13.51,61.99,45.70,6.51
Alagoas,34.89,11.59,50.50,33.70,6.31
